# Question 1: datetime Fundamentals and Time Series Indexing

This question focuses on datetime handling and time series indexing using patient vital signs data.

## Setup

In [1]:
import sys
!{sys.executable} -m pip install seaborn


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('default')
sns.set_style('whitegrid')

# Create output directory
os.makedirs('output', exist_ok=True)

## Part 1.1: Load and Explore Data

**Note:** This dataset contains realistic healthcare data characteristics:
- **200 patients** with daily vital signs over 1 year
- **Missing visits**: Patients miss approximately 5% of scheduled visits (realistic!)
- **Different start dates**: Not all patients start monitoring on January 1st (some join later)
- When selecting data by date ranges, you may find that some patients don't have data for certain periods - this is expected and realistic

In [3]:
# Load patient vital signs data
patient_vitals = pd.read_csv('data/patient_vitals.csv')

print("Patient vitals shape:", patient_vitals.shape)
print("Patient vitals columns:", patient_vitals.columns.tolist())

# Display sample data
print("\nPatient vitals sample:")
print(patient_vitals.head())
print("\nData summary:")
print(patient_vitals.describe())

# Check date range and missing data patterns
print(f"\nDate range: {patient_vitals['date'].min()} to {patient_vitals['date'].max()}")
print(f"Unique patients: {patient_vitals['patient_id'].nunique()}")
print(f"Total records: {len(patient_vitals)}")
print(f"Expected records (200 patients × 365 days): {200 * 365:,}")
print(f"Missing visits: ~{200 * 365 - len(patient_vitals):,} records")

Patient vitals shape: (18250, 7)
Patient vitals columns: ['date', 'patient_id', 'temperature', 'heart_rate', 'blood_pressure_systolic', 'blood_pressure_diastolic', 'weight']

Patient vitals sample:
         date patient_id  temperature  heart_rate  blood_pressure_systolic  \
0  2023-01-01      P0001    98.389672          71                      119   
1  2023-01-02      P0001    98.492046          67                      117   
2  2023-01-03      P0001    98.790163          70                      113   
3  2023-01-04      P0001    98.635781          74                      117   
4  2023-01-05      P0001    98.051660          67                      118   

   blood_pressure_diastolic     weight  
0                        84  68.996865  
1                        82  67.720215  
2                        78  67.846825  
3                        82  67.693993  
4                        83  68.228852  

Data summary:
        temperature    heart_rate  blood_pressure_systolic  \
count  182

## Part 1.2: datetime Operations

**TODO: Perform datetime operations**

In [4]:
import pandas as pd
import numpy as np
import os

# Make sure output folder exists
os.makedirs('output', exist_ok=True)

# Load data
patient_vitals = pd.read_csv('data/patient_vitals.csv')

# Convert 'date' column to datetime
patient_vitals['date'] = pd.to_datetime(patient_vitals['date'])

# Extract year, month, day components
patient_vitals['year'] = patient_vitals['date'].dt.year
patient_vitals['month'] = patient_vitals['date'].dt.month
patient_vitals['day'] = patient_vitals['date'].dt.day

# Calculate days since first measurement per patient
patient_vitals['days_since_start'] = patient_vitals.groupby('patient_id')['date'].transform(lambda x: (x - x.min()).dt.days)

# Optionally: create clinic visit date ranges
clinic_dates = pd.date_range(start=patient_vitals['date'].min(), end=patient_vitals['date'].max())
daily_range = pd.date_range(start=patient_vitals['date'].min(), end=patient_vitals['date'].max(), freq='D')
weekly_range = pd.date_range(start=patient_vitals['date'].min(), end=patient_vitals['date'].max(), freq='W-MON')
monthly_range = pd.date_range(start=patient_vitals['date'].min(), end=patient_vitals['date'].max(), freq='MS')

# Analyze visit patterns
patient_dates_set = set(patient_vitals['date'].dt.date)
clinic_dates_set = set(clinic_dates.date)
visits_on_clinic_days = len(patient_dates_set & clinic_dates_set)
visits_on_weekends = len(patient_dates_set) - visits_on_clinic_days
print(f"Visits on clinic business days: {visits_on_clinic_days}")
print(f"Visits on weekends: {visits_on_weekends}")
print(f"Total unique visit dates: {len(patient_dates_set)}")

# Save results to CSV with required columns, including 'date' first
cols_to_save = ['date', 'patient_id', 'year', 'month', 'day', 'days_since_start']

# Add one original vital sign if exists
for col in ['temperature', 'heart_rate', 'blood_pressure_systolic', 'blood_pressure_diastolic']:
    if col in patient_vitals.columns:
        cols_to_save.append(col)
        break

datetime_analysis = patient_vitals[cols_to_save].copy()
datetime_analysis.to_csv('output/q1_datetime_analysis.csv', index=False)

print("q1_datetime_analysis.csv saved with columns:", datetime_analysis.columns.tolist())


Visits on clinic business days: 365
Visits on weekends: 0
Total unique visit dates: 365
q1_datetime_analysis.csv saved with columns: ['date', 'patient_id', 'year', 'month', 'day', 'days_since_start', 'temperature']


## Part 1.3: Time Zone Handling

**TODO: Handle time zones**

In [7]:
import pandas as pd
import os

# Make sure output folder exists
os.makedirs('output', exist_ok=True)

# Create timezone-aware datetime
utc_time = pd.Timestamp.now(tz='UTC')
eastern_time = utc_time.tz_convert('US/Eastern')
print("Now UTC:", utc_time)
print("Now Eastern:", eastern_time)

# Create timezone-aware DataFrame from patient_vitals
# Ensure 'date' column exists
patient_vitals_tz = patient_vitals.copy()
patient_vitals_tz['date'] = pd.to_datetime(patient_vitals_tz['date'])
patient_vitals_tz = patient_vitals_tz.set_index('date')
patient_vitals_tz.index = patient_vitals_tz.index.tz_localize('UTC')
print("\nUTC Timezone-aware datetimes:")
print(patient_vitals_tz.head())

# Convert to US/Eastern timezone
patient_vitals_tz_eastern = patient_vitals_tz.copy()
patient_vitals_tz_eastern.index = patient_vitals_tz_eastern.index.tz_convert('US/Eastern')
print("\nConverted to US/Eastern timezone:")
print(patient_vitals_tz_eastern.head())

# Handle daylight saving time transitions
dst_date_utc = pd.Timestamp('2023-03-12 10:00:00', tz='UTC')
dst_time_eastern = dst_date_utc.tz_convert('US/Eastern')
print("\nDST Transition Example:")
print("UTC Time:", dst_date_utc, "-> Eastern Time:", dst_time_eastern)

# Document timezone operations
timezone_report = """Timezone Operations Report

1. Original timezone:
The original patient_vitals dataset contained naive datetime objects (no timezone info).

2. Localization method:
Naive datetime indices were localized to UTC using tz_localize('UTC') to make them timezone-aware.

3. Conversion:
The UTC timestamps were then converted to US/Eastern using tz_convert('US/Eastern') for local interpretation.

4. DST handling:
Using UTC as the base timezone avoids ambiguity caused by daylight saving time transitions. For example, March 12, 2023, 10:00:00 UTC converts to 06:00:00 in US/Eastern before the DST jump. Storing timestamps in UTC ensures calculations, comparisons, and merging across sites are unambiguous.

5. Example:
Original UTC timestamp: {dst_date_utc}
Converted Eastern timestamp: {dst_time_eastern}

"""

# Save report
with open('output/q1_timezone_report.txt', 'w') as f:
    f.write(timezone_report)

print("Timezone report saved to 'output/q1_timezone_report.txt'")


Now UTC: 2025-11-12 22:34:34.341644+00:00
Now Eastern: 2025-11-12 17:34:34.341644-05:00

UTC Timezone-aware datetimes:
                          patient_id  temperature  heart_rate  \
date                                                            
2023-01-01 00:00:00+00:00      P0001    98.389672          71   
2023-01-02 00:00:00+00:00      P0001    98.492046          67   
2023-01-03 00:00:00+00:00      P0001    98.790163          70   
2023-01-04 00:00:00+00:00      P0001    98.635781          74   
2023-01-05 00:00:00+00:00      P0001    98.051660          67   

                           blood_pressure_systolic  blood_pressure_diastolic  \
date                                                                           
2023-01-01 00:00:00+00:00                      119                        84   
2023-01-02 00:00:00+00:00                      117                        82   
2023-01-03 00:00:00+00:00                      113                        78   
2023-01-04 00:00:00+00:00

## Submission Checklist

Before moving to Question 2, verify you've created:

- [ ] `output/q1_datetime_analysis.csv` - datetime analysis results
- [ ] `output/q1_timezone_report.txt` - timezone handling report
